# Lingi7 Full Platform — Colab Test

Runs **all 3 apps** on Google Colab (T4 GPU):
- **Lingi7 Django** (port 8000) — e-commerce + fintech core
- **Enrichment FastAPI** (port 8001) — AI catalog enrichment
- **Assistant Chain Server** (port 8002) — AI shopping assistant

Supporting stubs on ports 8010-8012 (catalog retriever, memory, guardrails).

**Prerequisites:** Runtime > Change runtime type > **T4 GPU**

## 1. Install system dependencies

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1 && echo 'System deps installed'

## 2. Install Ollama + pull models

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
print("Ollama server started!")

!ollama pull qwen2.5:7b
!ollama pull llava:7b
print("Models pulled!")
!ollama list

## 3. Install Python dependencies (all 3 apps)

In [ ]:
# Django core deps
!pip install -q Django==4.2.16 djangorestframework==3.15.2 django-cors-headers==4.4.0 \
  djangorestframework-simplejwt==5.3.1 drf-spectacular==0.29.0 dj-database-url==2.2.0 \
  redis==5.0.8 celery==5.4.0 django-celery-beat==2.7.0 \
  django-celery-results==2.5.1 kombu==5.3.4 boto3==1.35.0 django-storages==1.14.4 \
  Pillow==10.4.0 django-otp==1.5.1 qrcode==8.0 cryptography==43.0.1 \
  httpx==0.27.2 python-decouple==3.8 phonenumbers==8.13.45 django-filter==24.3 \
  django-extensions==3.2.3 scikit-learn==1.5.2 xgboost==2.1.1 joblib==1.4.2 \
  numpy==1.26.4 pandas==2.2.3

# Enrichment deps
!pip install -q fastapi uvicorn openai pydantic python-multipart aiofiles \
  python-dotenv pyyaml pypdf

# Assistant deps
!pip install -q langgraph langchain langchain-core langchain-text-splitters \
  langgraph-prebuilt langgraph-sdk langsmith tenacity tqdm

print("All dependencies installed!")

## 4. Clone the repo

In [ ]:
import os

if os.path.exists("/content/lingi7"):
    !cd /content/lingi7 && git checkout -- . && git pull origin main
else:
    !git clone https://github.com/rager1904/lingi7.git
os.chdir("/content/lingi7")

print(f"CWD: {os.getcwd()}")
print(f"lingi7 exists: {os.path.exists('lingi7/manage.py')}")
print(f"enrichment exists: {os.path.exists('enrichment/src/backend/main.py')}")
print(f"assistant exists: {os.path.exists('assistant/chain_server/src/main.py')}")

## 5. (Skipped — using SQLite for Colab)

In [ ]:
print("Using SQLite — no PostgreSQL needed for Colab test")

## 6. Start Redis

In [ ]:
try:
    !redis-server --daemonize yes
    print("Redis started!")
except:
    print("Redis not available - skipping (only needed for Celery)")

## 7. Write environment variables

In [ ]:
import os

os.environ["SECRET_KEY"] = "colab-test-secret-key-not-for-production"
os.environ["DEBUG"] = "True"
os.environ["DJANGO_ALLOWED_HOSTS"] = "localhost,127.0.0.1"
os.environ["REDIS_URL"] = "redis://localhost:6379/0"
os.environ["CELERY_BROKER_URL"] = "redis://localhost:6379/1"
os.environ["CELERY_RESULT_BACKEND"] = "redis://localhost:6379/2"
os.environ["API_KEY"] = "ollama"
os.environ["LLM_API_KEY"] = "ollama"

os.makedirs("/content/lingi7/lingi7/static", exist_ok=True)
os.makedirs("/content/data/policies", exist_ok=True)

print("Environment variables set!")

## 8. Write config files for all services

In [ ]:
import os

REPO = "/content/lingi7"

# Enrichment config
os.makedirs(f"{REPO}/enrichment/shared/config", exist_ok=True)
enrichment_lines = [
    'vlm:',
    '  url: "http://localhost:11434/v1"',
    '  model: "llava:7b"',
    '  max_tokens: 1024',
    '  temperature: 0.1',
    '',
    'llm:',
    '  url: "http://localhost:11434/v1"',
    '  model: "qwen2.5:7b"',
    '  max_tokens: 2048',
    '  temperature: 0.3',
    '',
    'embeddings:',
    '  url: "http://localhost:11434/v1"',
    '  model: "qwen2.5:7b"',
    '',
    'flux:',
    '  url: "http://localhost:8003/v1/infer"',
    '',
    'trellis:',
    '  url: "http://localhost:8004/v1/infer"',
    '',
    'milvus:',
    '  host: "localhost"',
    '  port: 19530',
    '  collection: "policy_chunks"',
    '  alias: "policy_library"',
    '',
    'product_manual:',
    '  chunk_size_words: 250',
    '  chunk_overlap_words: 50',
    '  top_k_per_query: 3',
    '  min_relevance_score: 0.25',
    '',
    'policy_library:',
    '  storage_dir: "/content/data/policies"',
    '  db_path: "/content/data/policies/library.db"',
    '  top_k: 8',
    '  min_relevance_score: 0.3',
    '  max_policy_text_chars: 12000',
    '  normalization_max_tokens: 2048',
    '  classification_max_tokens: 1024',
    '  embedding_batch_size: 128',
    '  embedding_dim: 384',
    '',
    'locales:',
    '  default: "en-US"',
    '  supported:',
    '    - "en-US"',
    '    - "en-GB"',
    '    - "en-AU"',
    '    - "en-CA"',
    '    - "es-ES"',
    '    - "es-MX"',
    '    - "es-AR"',
    '    - "es-CO"',
    '    - "fr-FR"',
    '    - "fr-CA"',
]
with open(f"{REPO}/enrichment/shared/config/config.yaml", "w") as f:
    f.write("\n".join(enrichment_lines))

# Assistant chain_server config
os.makedirs(f"{REPO}/assistant/shared/configs/chain_server", exist_ok=True)
routing = (
    "You are a retail store assistant that routes customer queries to the appropriate specialist.\n"
    "Available specialists:\n"
    "1. Cart Manager (cart_node): Handles adding/removing items from the cart\n"
    "2. Product Finder (search): Discovers NEW products in the store catalog\n"
    "3. General Assistant (chatter): Answers questions about SPECIFIC products and handles general conversation\n"
    "CART OPERATIONS -> cart_node\n"
    "NEW PRODUCT DISCOVERY -> search\n"
    "SPECIFIC PRODUCT QUESTIONS -> chatter\n"
    "GENERAL CONVERSATION -> chatter\n"
    "Always respond with exactly one of: cart_node, search, or chatter."
)
chain_lines = [
    'llm_port: "http://localhost:11434/v1"',
    'llm_name: "qwen2.5:7b"',
    'retriever_port: "http://localhost:8010"',
    'memory_port: "http://localhost:8011"',
    'rails_port: "http://localhost:8012"',
    'routing_prompt: |',
]
for line in routing.split("\n"):
    chain_lines.append("  " + line)
chain_lines += [
    'chatter_prompt: "You are a helpful shopping assistant. Be concise and helpful."',
    'categories: ["bag", "sunglasses", "dress", "skirt", "top blouse sweater", "shoes", "earrings", "bracelet", "necklace"]',
    'agent_choices: ["cart", "retriever", "chatter"]',
    'memory_length: 16384',
    'top_k_retrieve: 4',
    'multimodal: true',
    'unsafe_message: "Sorry, I am a shopping assistant that specializes in apparel."',
]
with open(f"{REPO}/assistant/shared/configs/chain_server/config.yaml", "w") as f:
    f.write("\n".join(chain_lines))

print("All config files written!")

## 9. Create data directories

In [ ]:
import os
os.makedirs("/content/data/policies", exist_ok=True)
print("Data directories ready")

## 10. Create stub services (catalog retriever, guardrails)

In [ ]:
import os

os.makedirs("/content/stubs", exist_ok=True)

catalog_stub_lines = [
    'from fastapi import FastAPI',
    'from pydantic import BaseModel',
    'from typing import List, Dict, Any',
    'import time',
    '',
    'app = FastAPI()',
    '',
    'class TextQueryRequest(BaseModel):',
    '    text: List[str] = []',
    '    categories: List[str] = []',
    '    filters: Dict[str, Any] = {}',
    '    k: int = 4',
    '',
    'class ImageQueryRequest(BaseModel):',
    '    text: List[str] = []',
    '    image_base64: str = ""',
    '    categories: List[str] = []',
    '    filters: Dict[str, Any] = {}',
    '    k: int = 4',
    '',
    '@app.post("/query/text")',
    'async def query_text(req: TextQueryRequest):',
    '    return {"texts": [], "ids": [], "similarities": [], "names": [], "images": []}',
    '',
    '@app.post("/query/image")',
    'async def query_image(req: ImageQueryRequest):',
    '    return {"texts": [], "ids": [], "similarities": [], "names": [], "images": []}',
    '',
    '@app.get("/health")',
    'async def health():',
    '    return {"status": "healthy", "stub": True}',
]
with open("/content/stubs/catalog_stub.py", "w") as f:
    f.write("\n".join(catalog_stub_lines))

guardrails_stub_lines = [
    'from fastapi import FastAPI',
    'from pydantic import BaseModel',
    '',
    'app = FastAPI()',
    '',
    'class QueryRequest(BaseModel):',
    '    user_id: int = 0',
    '    query: str = ""',
    '',
    '@app.post("/rail/input/check")',
    'async def check_input(request: QueryRequest):',
    '    return {"is_safe": True, "message": ""}',
    '',
    '@app.post("/rail/output/check")',
    'async def check_output(request: QueryRequest):',
    '    return {"is_safe": True, "message": ""}',
    '',
    '@app.post("/rail/input/timing")',
    'async def timing_input(request: QueryRequest):',
    '    return {"is_safe": True, "message": "", "timings": []}',
    '',
    '@app.post("/rail/output/timing")',
    'async def timing_output(request: QueryRequest):',
    '    return {"is_safe": True, "message": "", "timings": []}',
]
with open("/content/stubs/guardrails_stub.py", "w") as f:
    f.write("\n".join(guardrails_stub_lines))

print("Stub services created!")

## 11. Start all services

In [ ]:
import subprocess, time, sys, os, signal

REPO = "/content/lingi7"
os.environ["API_KEY"] = "ollama"
os.environ["LLM_API_KEY"] = "ollama"
os.environ["SILENCED_SYSTEM_CHECKS"] = "staticfiles.W004,models.E012"

# Kill ALL existing uvicorn/manage.py processes
os.system("pkill -9 -f uvicorn || true")
os.system("pkill -9 -f 'manage.py runserver' || true")
os.system("pkill -9 -f 'python.*8000' || true")
os.system("pkill -9 -f 'python.*8001' || true")
os.system("pkill -9 -f 'python.*8002' || true")
time.sleep(3)
print("Killed leftover processes")

# Delete stale SQLite DB
import glob as _glob
seen = set()
for f in _glob.glob(f"{REPO}/**/db.sqlite3", recursive=True):
    if f not in seen:
        seen.add(f)
        try:
            os.remove(f)
            print(f"Removed stale DB: {f}")
        except FileNotFoundError:
            pass

services = {}

def start_service(name, cmd, cwd=None, wait=3):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd=cwd)
    services[name] = proc
    time.sleep(wait)
    if proc.poll() is not None:
        output = proc.stdout.read().decode()[:800]
        print(f"  {name}: FAILED - {output}")
    else:
        print(f"  {name}: running (PID {proc.pid})")
    return proc

print("Starting services...")

start_service("catalog-retriever", [sys.executable, "-m", "uvicorn", "catalog_stub:app", "--host", "0.0.0.0", "--port", "8010"], "/content/stubs", 2)
start_service("guardrails", [sys.executable, "-m", "uvicorn", "guardrails_stub:app", "--host", "0.0.0.0", "--port", "8012"], "/content/stubs", 2)
start_service("memory-retriever", [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8011"], f"{REPO}/assistant/memory_retriever/src", 2)
start_service("django", [sys.executable, "manage.py", "runserver", "0.0.0.0:8000", "--noreload", "--skip-checks"], f"{REPO}/lingi7", 3)
start_service("enrichment", [sys.executable, "-m", "uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8001"], f"{REPO}/enrichment/src", 3)

env = os.environ.copy()
env["SHARED_CONFIG_ROOT"] = f"{REPO}/assistant/shared/configs"
chain_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "src.main:app", "--host", "0.0.0.0", "--port", "8002"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    cwd=f"{REPO}/assistant/chain_server", env=env
)
services["chain-server"] = chain_proc
time.sleep(3)
if chain_proc.poll() is not None:
    print(f"  chain-server: FAILED - {chain_proc.stdout.read().decode()[:800]}")
else:
    print(f"  chain-server: running (PID {chain_proc.pid})")

print(f"\nAll {len(services)} services launched!")

## 12. Run Django migrations and create superuser

In [ ]:
import subprocess, os

# Ensure no stale DATABASE_URL causes dj-database-url to pick PostgreSQL
os.environ.pop("DATABASE_URL", None)

r = subprocess.run(
    ["python", "manage.py", "migrate", "--run-syncdb", "--skip-checks"],
    capture_output=True, text=True,
    cwd="/content/lingi7/lingi7"
)
print(r.stdout[-2000:] if r.stdout else '')
if r.returncode != 0:
    print('ERRORS:')
    print(r.stderr[-2000:] if r.stderr else '')
else:
    print('Migrations OK')

script = (
    'from apps.users.models import User\n'
    'if not User.objects.filter(phone_number="+260977777777").exists():\n'
    '    User.objects.create_superuser(phone_number="+260977777777", password="admin123", first_name="Super", last_name="Admin")\n'
    '    print("Superuser created")\n'
    'else:\n'
    '    print("Superuser already exists")\n'
)
r2 = subprocess.run(
    ["python", "manage.py", "shell", "-c", script],
    capture_output=True, text=True,
    cwd="/content/lingi7/lingi7"
)
print(r2.stdout.strip() if r2.stdout else '')
if r2.returncode != 0:
    print(r2.stderr[-500:] if r2.stderr else '')

## 13. Set service URLs (localhost)

In [ ]:
DJANGO_URL = "http://localhost:8000"
ENRICH_URL = "http://localhost:8001"
ASSISTANT_URL = "http://localhost:8002"

print(f"Django:    {DJANGO_URL}")
print(f"Enrichment: {ENRICH_URL}")
print(f"Assistant: {ASSISTANT_URL}")

## 14. Health checks

In [ ]:
import requests

for name, port, path in [("Django", 8000, "/health/"), ("Enrichment", 8001, "/health"),
                    ("Chain Server", 8002, "/"), ("Catalog Retriever", 8010, "/health"),
                    ("Memory Retriever", 8011, "/"), ("Guardrails", 8012, "/")]:
    try:
        r = requests.get(f"http://localhost:{port}{path}", timeout=5)
        print(f"  {name}: {r.status_code}")
    except Exception as e:
        print(f"  {name}: DOWN ({e})")

---
# WORKFLOW TESTS

In [ ]:
# Diagnostic: test Django from inside the same process
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'lingi7'))
os.chdir(os.path.join(os.getcwd(), 'lingi7')) if os.path.exists('manage.py') else None
import django
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'config.settings.base')
django.setup()
print('Django setup OK')

import json
from django.test.client import Client
c = Client()

print('=== Health check ===')
resp = c.get('/health/')
print(f'Health: {resp.status_code} {resp.content.decode()[:200]}')

print('\n=== Register ===')
resp = c.post('/api/v1/auth/register/', data=json.dumps({
    'phone_number': '+260971234568', 'password': 'TestPass123!',
    'password_confirm': 'TestPass123!', 'first_name': 'Test',
    'last_name': 'User', 'consent_given': True
}), content_type='application/json')
print(f'Register: {resp.status_code}')
print(resp.content.decode()[:1000])

print('\n=== Login ===')
resp = c.post('/api/v1/auth/token/', data=json.dumps({
    'phone_number': '+260977777777', 'password': 'admin123'
}), content_type='application/json')
print(f'Login: {resp.status_code}')
print(resp.content.decode()[:1000])

## 15. Test: Django — User Registration

In [ ]:
import requests, json

r = requests.post(f"{DJANGO_URL}/api/v1/auth/register/", json={
    "phone_number": "+260971234567",
    "password": "TestPass123!",
    "password_confirm": "TestPass123!",
    "first_name": "Test",
    "last_name": "User",
    "consent_given": True
})
print(f"Register: {r.status_code}")
print(json.dumps(r.json(), indent=2))

## 16. Test: Django — Login (JWT)

In [ ]:
r = requests.post(f"{DJANGO_URL}/api/v1/auth/token/", json={
    "phone_number": "+260977777777",
    "password": "admin123"
})
print(f"Login: {r.status_code}")
login_data = r.json()
print(json.dumps(login_data, indent=2)[:500])

TOKEN = login_data.get("access", "")
AUTH_HEADERS = {"Authorization": f"Bearer {TOKEN}"}
print(f"\nToken obtained: {TOKEN[:30]}...")

## 17. Test: Django — Get user profile

In [ ]:
r = requests.get(f"{DJANGO_URL}/api/v1/auth/me/", headers=AUTH_HEADERS)
print(f"Profile: {r.status_code}")
print(json.dumps(r.json(), indent=2)[:500])

## 18. Test: Django — API schema (Swagger)

In [ ]:
r = requests.get(f"{DJANGO_URL}/api/schema/swagger/", timeout=5)
print(f"Swagger UI: {r.status_code}")
print(f"Open at: {DJANGO_URL}/api/schema/swagger/")

## 19. Test: Enrichment — VLM Analysis

In [ ]:
from google.colab import files
from PIL import Image
import io

print("Upload a product image (or skip for sample):")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    IMAGE_BYTES = uploaded[filename]
    CONTENT_TYPE = "image/jpeg" if ".jpg" in filename.lower() or ".jpeg" in filename.lower() else "image/png"
else:
    img = Image.new("RGB", (400, 400), color=(255, 100, 50))
    img.save("/content/sample.jpg", "JPEG")
    IMAGE_BYTES = open("/content/sample.jpg", "rb").read()
    CONTENT_TYPE = "image/jpeg"

print(f"Image ready: {len(IMAGE_BYTES)} bytes")

In [ ]:
import json, time

print("Running VLM analysis... (may take 3-5 min on free T4 GPU)")
start = time.time()
try:
    r = requests.post(
        f"{ENRICH_URL}/vlm/analyze",
        files={"image": ("product.jpg", IMAGE_BYTES, CONTENT_TYPE)},
        data={"locale": "en-US"},
        timeout=360,
    )
    elapsed = time.time() - start
    print(f"Status: {r.status_code} ({elapsed:.1f}s)")
    vlm_result = r.json() if r.status_code == 200 else None
    if vlm_result:
        print(json.dumps(vlm_result, indent=2, ensure_ascii=False))
    else:
        print(f"Error: {r.text[:300]}")
except requests.exceptions.ReadTimeout:
    elapsed = time.time() - start
    print(f"VLM timed out after {elapsed:.0f}s — expected on free T4 GPU")
    print("Skipping VLM-dependent tests. Other services still work.")
    vlm_result = None

## 20. Test: Enrichment — FAQ Generation

In [ ]:
if vlm_result:
    r = requests.post(f"{ENRICH_URL}/vlm/faqs", data={
        "title": vlm_result.get("title", ""),
        "description": vlm_result.get("description", ""),
        "categories": json.dumps(vlm_result.get("categories", [])),
        "tags": json.dumps(vlm_result.get("tags", [])),
        "colors": json.dumps(vlm_result.get("colors", [])),
        "locale": "en-US",
    }, timeout=120)
    print(f"FAQ: {r.status_code}")
    if r.status_code == 200:
        print(json.dumps(r.json(), indent=2))
        faqs_result = r.json()
    else:
        print(r.text[:300])
        faqs_result = None
else:
    print("Skipping - no VLM result")
    faqs_result = None

## 21. Test: Enrichment — Protocol Schemas

In [ ]:
if vlm_result:
    faqs_list = faqs_result.get("faqs", []) if faqs_result else []
    r = requests.post(f"{ENRICH_URL}/protocols/generate", data={
        "title": vlm_result.get("title", ""),
        "description": vlm_result.get("description", ""),
        "categories": json.dumps(vlm_result.get("categories", [])),
        "tags": json.dumps(vlm_result.get("tags", [])),
        "colors": json.dumps(vlm_result.get("colors", [])),
        "faqs": json.dumps(faqs_list),
        "locale": "en-US",
    }, timeout=120)
    print(f"Protocols: {r.status_code}")
    if r.status_code == 200:
        p = r.json()
        print("ACP keys:", list(p.get("acp", {}).keys()))
        print("UCP keys:", list(p.get("ucp", {}).keys()))
    else:
        print(r.text[:300])

## 22. Test: Assistant — Shopping Query

In [ ]:
import json, time

print("Testing assistant with a shopping query...")
start = time.time()
try:
    r = requests.post(f"{ASSISTANT_URL}/query/timing", json={
        "user_id": 1,
        "query": "Show me some dresses"
    }, timeout=180)
    elapsed = time.time() - start
    print(f"Status: {r.status_code} ({elapsed:.1f}s)")
    if r.status_code == 200:
        print(json.dumps(r.json(), indent=2)[:1000])
    else:
        print(r.text[:500])
except requests.exceptions.ReadTimeout:
    print(f"Assistant timed out after {time.time()-start:.0f}s — Qwen 7b is slow on T4")

## 23. Test: Assistant — Streaming Query

In [ ]:
print("Testing streaming assistant query...")
start = time.time()
try:
    r = requests.post(f"{ASSISTANT_URL}/query/stream", json={
        "user_id": 1,
        "query": "What bags do you have?"
    }, stream=True, timeout=180)

    print(f"Status: {r.status_code}")
    for line in r.iter_lines():
        if line:
            print(line.decode())
    elapsed = time.time() - start
    print(f"\n({elapsed:.1f}s)")
except requests.exceptions.ReadTimeout:
    print(f"Streaming timed out after {time.time()-start:.0f}s")

## 24. Test: Assistant — Cart Operation

In [ ]:
try:
    r = requests.post(f"{ASSISTANT_URL}/query/timing", json={
        "user_id": 1,
        "query": "Add a blue dress to my cart"
    }, timeout=180)
    print(f"Cart add: {r.status_code}")
    if r.status_code == 200:
        print(json.dumps(r.json(), indent=2)[:1000])

    r = requests.post(f"{ASSISTANT_URL}/query/timing", json={
        "user_id": 1,
        "query": "What's in my cart?"
    }, timeout=180)
    print(f"\nCart view: {r.status_code}")
    if r.status_code == 200:
        print(json.dumps(r.json(), indent=2)[:1000])
except requests.exceptions.ReadTimeout:
    print(f"Assistant timed out — Qwen 7b is slow on T4")

## 25. Test: Memory Retriever

In [ ]:
r = requests.get("http://localhost:8011/user/1/context", timeout=10)
print(f"Memory: {r.status_code}")
print(json.dumps(r.json(), indent=2)[:500])

## 26. Check server logs

In [ ]:
for name, proc in services.items():
    status = "running" if proc.poll() is None else f"stopped (rc={proc.poll()})"
    print(f"  {name}: {status}")

## Summary

### Services running
- **Django** (port 8000) — user auth, products, orders, escrow, fraud, payments
- **Enrichment** (port 8001) — VLM analysis, FAQ generation, protocol schemas
- **Chain Server** (port 8002) — AI shopping assistant with LangGraph
- **Catalog Retriever stub** (port 8010) — returns empty results (needs Milvus + BGE/CLIP for real data)
- **Memory Retriever** (port 8011) — user context persistence
- **Guardrails stub** (port 8012) — pass-through (needs Llama Guard for real safety)

### What works end-to-end
- User registration + JWT login + profile
- VLM image analysis -> enriched product data
- FAQ generation from enriched data
- ACP/UCP protocol schema export
- Assistant query routing (planner -> agents)
- Cart operations via assistant
- Streaming responses

### What is stubbed
- Catalog retrieval (no Milvus/BGE/CLIP on Colab)
- Content safety guardrails (no Llama Guard on Colab)
- Image variation (FLUX) and 3D generation (TRELLIS)

### Note
If Django 404s on auth endpoints, the URL namespace may differ. Check `{DJANGO_URL}/api/schema/swagger/` for available routes.